# MTKD4SER: Multi-Teacher Knowledge Distillation for Speech Emotion Recognition

This notebook provides a complete pipeline to train and evaluate the **MTKD4SER** model on **Google Colab (Free T4 GPU / 15GB VRAM)**.

### Key Highlights:
- **Language-Aware Dynamic Routing**: Oracle Language routing, Uniform, Cosine similarity, and Confidence routing.
- **VRAM Optimized**: Dynamic dual-teacher setup (English + French) avoiding Finnish teacher duplication, safely running on T4 GPU.
- **Hugging Face Hub Streaming & Arrow Support**: Direct loading from Hugging Face Hub (zero-copy Arrow tables) without Google Drive upload bottlenecks.
- **Speaker-Independent Splits**: 0% speaker leakage across Train, Dev, and Test sets.

## 1. Verify GPU Hardware
Colab provides Nvidia T4 (15 GB VRAM). Ensure CUDA is available.

In [ ]:
!nvidia-smi

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")

## 2. Mount Google Drive (Optional)
Mount Google Drive if you wish to persist trained checkpoints (`/content/checkpoints`) across Colab sessions. If using Hugging Face Hub for datasets and Weights & Biases for artifact tracking, mounting Drive is optional.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Clone Repository & Install Dependencies
Replace the repository URL below with your GitHub repository.

In [ ]:
# Clone the repository (or upload project folder)
!git clone https://github.com/nhatientri/mtkd4ser_phase1.git
%cd mtkd4ser_phase1

# Install project dependencies
!pip install -q transformers datasets torchaudio soundfile librosa scikit-learn tqdm tensorboard accelerate pyarrow wandb huggingface_hub pytest tabulate

# Quick self-check: routing / loss / batching unit tests (a few seconds)
!python -m pytest -q tests/test_distill.py tests/test_batching_metrics.py

### Hugging Face Authentication & Hub Setup (Required for Private Repositories)
Setting your `HF_TOKEN` authenticates access to your Hugging Face dataset repository (e.g. `nhatientri/mtkd4ser-phase1`) and avoids rate limits.
You can store `HF_TOKEN` in Colab Secrets (the 🔑 icon on the left sidebar) or log in directly via `huggingface_hub.login()`.

In [ ]:
import os
from huggingface_hub import login

try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
    if hf_token:
        os.environ["HF_TOKEN"] = hf_token
        login(token=hf_token)
        print("✓ Authenticated to Hugging Face via Colab Secrets.")
    else:
        print("ℹ No HF_TOKEN detected in Colab Secrets.")
        print("  If using a private Hugging Face dataset, log in interactively or set your token below.")
        # login()
except Exception as e:
    print(f"ℹ Hugging Face authentication note: {e}")
    # login(token="hf_...")


### Weights & Biases (W&B) Authentication (Recommended)
Logging to [Weights & Biases](https://wandb.ai) gives you live remote loss curves, teacher routing weight distributions, confusion matrices, and automatic cloud model artifact versioning.

Store your `WANDB_API_KEY` in Colab Secrets (the 🔑 icon on the left sidebar) or log in interactively.

In [ ]:
import os
try:
    from google.colab import userdata
    wandb_key = userdata.get("WANDB_API_KEY")
    if wandb_key:
        import wandb
        wandb.login(key=wandb_key)
        print("✓ W&B successfully authenticated from Colab Secrets.")
    else:
        print("ℹ No WANDB_API_KEY detected in Colab Secrets. You can log in interactively:")
        import wandb
        wandb.login()
except Exception as e:
    print(f"ℹ Interactive login fallback: {e}")
    # import wandb
    # wandb.login()


## 4. Dataset Preparation

### Option A: Hugging Face Hub (Recommended - Zero Drive Storage Required)
Load pre-processed datasets directly from your Hugging Face Hub repository (`ngunghoc/mtkd4ser-phase1`).

> **Why your Hugging Face repository looks like this:**
> On Hugging Face Hub, multi-subset datasets store each configuration as a folder at the root of the repository (`cafe/`, `combined/`, `iemocap_session1/`, etc.). They are **not** supposed to be under `data/hf_datasets/` on the Hub.
> When you call `load_dataset('ngunghoc/mtkd4ser-phase1', 'combined')`, Hugging Face automatically reads the `combined/` configuration folder.
>
> ⚠️ **Private Repository Authentication:** Because `ngunghoc/mtkd4ser-phase1` is private, make sure you ran the Hugging Face Authentication cell in Step 3 above with your `HF_TOKEN`.

### Option B: Extract Pre-packaged Archive from Google Drive (Alternative / Offline)
If you uploaded `hf_data.zip` to your Google Drive, you can extract it locally into `./data/hf_datasets/`.

In [ ]:
# ============================================================================
# OPTION A: Set Hugging Face Dataset Repository (Recommended)
# ============================================================================
HF_DATASET_REPO = "ngunghoc/mtkd4ser-phase1"
os.environ["HF_DATASET_REPO"] = HF_DATASET_REPO
print(f"✓ Active Hugging Face Dataset Repository: {HF_DATASET_REPO}")


In [ ]:
# ============================================================================
# OPTION B: Extract Pre-packaged Archive from Google Drive (Alternative)
# ============================================================================
# Uncomment if you prefer extracting local hf_data.zip from Drive instead of HF Hub:
# !unzip -q /content/drive/MyDrive/hf_data.zip -d ./

import os
if os.path.exists("data/hf_datasets"):
    print("✓ Local Hugging Face Arrow datasets found:", os.listdir("data/hf_datasets"))
else:
    print(f"ℹ No local data/hf_datasets found; training will stream directly from Hugging Face: {os.environ.get('HF_DATASET_REPO')}")


### Quick Dataset Verification
Test that the dataset loads properly from Hugging Face Hub (or local cache):

In [ ]:
import os
from mtkd4ser.data import combined

hf_repo = os.environ.get("HF_DATASET_REPO", "ngunghoc/mtkd4ser-phase1")
print(f"Testing dataset loading (Repo: {hf_repo})...")

ds = combined(session_en=1, fold_fr=1, hf_repo=hf_repo)
print(ds)
print("Train features:", ds["train"].features)

## 5. Train Monolingual Teacher Models
Before running MTKD you need fine-tuned (FT) monolingual teachers for English and French.

> ⚠️ **Retrain teachers trained before the padding fix.** Older checkpoints were trained on inputs zero-padded to 20 s
> without an attention mask; the new code masks padding, so old teachers see a different input distribution.
>
> Run names now include backbone and seed, e.g. `FT_Monolingual_EN_S1_wav2vec2-base_seed42`, so the W&B artifact is
> `FT_Monolingual_EN_S1_wav2vec2-base_seed42-best:best`.
>
> To continue an interrupted run, add `--RESUME 1` (local `_last.pth`) or `--RESUME_CKPT <path or W&B artifact>`.
> Without these flags a run always starts fresh.

In [ ]:
# 1. English teacher (IEMOCAP, leave-one-session-out fold 1)
!python main.py --PARADIGM FT --LANGUAGE EN --SESSION 1 \
    --N_EPOCHS 12 --BATCH_SIZE 8 --LEARNING_RATE 3e-5 --SAVE_DIR /content/checkpoints --HF_DATASET_REPO "$HF_DATASET_REPO" --USE_WANDB 1 --WANDB_PROJECT mtkd4ser-phase1

In [ ]:
# 2. French teacher (CaFE fold 1)
!python main.py --PARADIGM FT --LANGUAGE FR --CAFE_FOLD 1 \
    --N_EPOCHS 12 --BATCH_SIZE 8 --LEARNING_RATE 3e-5 --SAVE_DIR /content/checkpoints --HF_DATASET_REPO "$HF_DATASET_REPO" --USE_WANDB 1 --WANDB_PROJECT mtkd4ser-phase1

# 3. Baseline MTKD must beat: multilingual fine-tuning on EN+FR, no teachers
!python main.py --PARADIGM FT --LINGUALITY Multilingual --SESSION 1 --CAFE_FOLD 1 \
    --N_EPOCHS 12 --BATCH_SIZE 8 --LEARNING_RATE 3e-5 --SAVE_DIR /content/checkpoints --HF_DATASET_REPO "$HF_DATASET_REPO" --USE_WANDB 1 --WANDB_PROJECT mtkd4ser-phase1

## 6. Multi-Teacher Knowledge Distillation (MTKD)
Train a student on EN+FR with both teachers.

- Teacher logits are computed **once** and cached in `/content/checkpoints/teacher_cache/`; the teachers are then removed
  from GPU memory, so each MTKD epoch costs about the same as fine-tuning the student alone.
- Routing is now **per sample**: with `language` routing a French clip is distilled only from the French teacher.
- The best epoch is chosen on the **mean of per-language dev UR** (so the small French set counts), and the test report
  shows UR for EN and FR separately.
- `--BALANCE_LANG 1` samples EN and FR clips equally often (FR is only ~8% of the training data).

In [ ]:
import os
# Teacher checkpoints: local paths or W&B artifacts ('name:alias', 'project/name:alias', 'entity/project/name:alias')
TEACHER_EN_CKPT = '/content/checkpoints/FT_Monolingual_EN_S1_wav2vec2-base_seed42.pth'
TEACHER_FR_CKPT = '/content/checkpoints/FT_Monolingual_FR_F1_wav2vec2-base_seed42.pth'
# e.g. from W&B:
# TEACHER_EN_CKPT = 'mtkd4ser-phase1/FT_Monolingual_EN_S1_wav2vec2-base_seed42-best:best'
# TEACHER_FR_CKPT = 'mtkd4ser-phase1/FT_Monolingual_FR_F1_wav2vec2-base_seed42-best:best'

os.environ['TEACHER_EN_CKPT'] = TEACHER_EN_CKPT
os.environ['TEACHER_FR_CKPT'] = TEACHER_FR_CKPT
print('EN teacher:', TEACHER_EN_CKPT)
print('FR teacher:', TEACHER_FR_CKPT)

In [ ]:
# ============================================================================
# (Optional) Verify & Pre-download Teacher Checkpoints from W&B
# ============================================================================
# Run this cell to ensure your W&B teacher artifacts are accessible and download
# them to /content/checkpoints ahead of MTKD training:
from mtkd4ser.checkpoints import resolve_checkpoint
WANDB_PROJECT = os.environ.get('WANDB_PROJECT', 'mtkd4ser-phase1')
WANDB_ENTITY = os.environ.get('WANDB_ENTITY', '')

print('Resolving English teacher checkpoint from W&B...')
resolved_en = resolve_checkpoint(TEACHER_EN_CKPT, save_dir='/content/checkpoints',
                                 default_project=WANDB_PROJECT, default_entity=WANDB_ENTITY or None)
print('Resolving French teacher checkpoint from W&B...')
resolved_fr = resolve_checkpoint(TEACHER_FR_CKPT, save_dir='/content/checkpoints',
                                 default_project=WANDB_PROJECT, default_entity=WANDB_ENTITY or None)

print(f'\n✓ English teacher ready at: {resolved_en}')
print(f'✓ French teacher ready at:  {resolved_fr}')


In [ ]:
# MTKD, oracle language routing, student = wav2vec2-base (same size as the teachers)
!python main.py --PARADIGM MTKD --LINGUALITY Multilingual --SESSION 1 --CAFE_FOLD 1 \
    --ROUTING language --N_EPOCHS 12 --BATCH_SIZE 8 --LEARNING_RATE 3e-5 \
    --TEACHER_EN_CKPT "$TEACHER_EN_CKPT" --TEACHER_FR_CKPT "$TEACHER_FR_CKPT" \
    --SAVE_DIR /content/checkpoints --HF_DATASET_REPO "$HF_DATASET_REPO" --USE_WANDB 1 --WANDB_PROJECT mtkd4ser-phase1

### Routing ablations and smaller students
Each routing mode is a separate run trained from scratch. For fast iteration, use a **smaller student** – distillation
only matches the 4 output logits, so the student does not have to share the teachers' architecture:

| `--BACKBONE` / `--NUM_LAYERS` | Params | Notes |
| :--- | ---: | :--- |
| `facebook/wav2vec2-base` | ~95M | same as teachers (paper setting) |
| `facebook/wav2vec2-base --NUM_LAYERS 6` | ~52M | pretrained, first 6 transformer layers |
| `facebook/wav2vec2-base --NUM_LAYERS 4` | ~38M | pretrained, first 4 transformer layers |
| `ntu-spml/distilhubert` | ~24M | 2 transformer layers, distilled from HuBERT |

Pick the winner on a small student, then confirm it with the full-size student.

In [ ]:
STUDENT = "--BACKBONE ntu-spml/distilhubert"   # or "--BACKBONE facebook/wav2vec2-base --NUM_LAYERS 4"

for routing in ["language", "uniform", "cosine", "confidence"]:
    !python main.py --PARADIGM MTKD --LINGUALITY Multilingual --SESSION 1 --CAFE_FOLD 1 \
        --ROUTING {routing} {STUDENT} --N_EPOCHS 12 --BATCH_SIZE 8 --LEARNING_RATE 5e-5 \
        --TEACHER_EN_CKPT "$TEACHER_EN_CKPT" --TEACHER_FR_CKPT "$TEACHER_FR_CKPT" \
        --SAVE_DIR /content/checkpoints --HF_DATASET_REPO "$HF_DATASET_REPO" --USE_WANDB 1 --WANDB_PROJECT mtkd4ser-phase1

# Same-size baseline without teachers, for a fair comparison
!python main.py --PARADIGM FT --LINGUALITY Multilingual --SESSION 1 --CAFE_FOLD 1 {STUDENT} \
    --N_EPOCHS 12 --BATCH_SIZE 8 --LEARNING_RATE 5e-5 --SAVE_DIR /content/checkpoints --HF_DATASET_REPO "$HF_DATASET_REPO" --USE_WANDB 1 --WANDB_PROJECT mtkd4ser-phase1

## 7. Evaluation and run comparison
`--TRAINING 0` evaluates a saved model on the test set (no teachers needed – the architecture is read from the checkpoint).
Every run also writes `/content/checkpoints/results/<run>.json` with test UR (overall / EN / FR), parameter count,
seconds per epoch, samples/s, peak GPU memory and test latency. `summarize_results.py` puts them in one table.

In [ ]:
STUDENT_CKPT = "/content/checkpoints/MTKD_language_Multilingual_ENFR_S1F1_wav2vec2-base_seed42.pth"
# or a W&B artifact: "mtkd4ser-phase1/MTKD_language_Multilingual_ENFR_S1F1_wav2vec2-base_seed42-best:best"

!python main.py --PARADIGM MTKD --LINGUALITY Multilingual --SESSION 1 --CAFE_FOLD 1 --TRAINING 0 \
    --STUDENT_CKPT "$STUDENT_CKPT" --SAVE_DIR /content/checkpoints --HF_DATASET_REPO "$HF_DATASET_REPO" --USE_WANDB 1 --WANDB_PROJECT mtkd4ser-phase1

# Compare all finished runs (accuracy + efficiency); --group = mean ± std over seeds / folds
!python summarize_results.py --SAVE_DIR /content/checkpoints

## 8. Monitor Training (Weights & Biases & TensorBoard)

- **Weights & Biases Dashboard**: Check your live training runs, confusion matrices, and model artifacts at [wandb.ai](https://wandb.ai).
- **TensorBoard**: Optionally visualize local logs via the embedded TensorBoard widget below.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/checkpoints/tb